In [1]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from RAG_pipeline import testing
from dotenv import load_dotenv
import pandas as pd
import os
load_dotenv()

D:\Temp\ipykernel_13420\3446135243.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
d:\User\Vakeesan's Pc\Download\DS Projects - Summer 2026\RAG & Eval Pipeline\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
os.getenv("OPENAI_API_KEY")

print("")

# Set Up

In [3]:
embedder = OpenAIEmbeddings(model='text-embedding-3-small')
encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
vectorstore = FAISS.load_local(r"D:\User\Vakeesan's Pc\Download\DS Projects - Summer 2026\RAG & Eval Pipeline\faiss_index", embedder, allow_dangerous_deserialization=True)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3771.95it/s]


In [ ]:
df = pd.read_csv("ragas_dataset.csv")
df.drop(columns=['id'], inplace=True)
df = df.rename(columns={"query": "user_input","ground_truth": 'reference'})
df.head()

,query,ground_truth
0,Who is the king of the Olympian gods?,Zeus is the king of the Olympian gods.
1,What is Zeus's domain and symbol?,Zeus rules the sky and thunder. His symbols ar...
2,Who are the children of Kronos and Rhea?,"Hestia, Demeter, Hera, Hades, Poseidon, and Zeus."
3,Why did Kronos swallow his children?,He feared a prophecy that one of his children ...
4,What is the Titanomachy?,The Titanomachy was a ten-year war between the...


In [ ]:
def createResponeReference(df, vectorstore, encoder):
    df[['response', 'retrieved_contexts']] = df['query'].apply(
        lambda x: pd.Series(testing(x, vectorstore, encoder))
    )


In [ ]:
#createResponeReference(df, vectorstore, encoder)
df.head()

,query,ground_truth,response,reference
0,Who is the king of the Olympian gods?,Zeus is the king of the Olympian gods.,The king of the Olympian gods is Zeus.,denying his divinity — King Pentheus of Thebes...
1,What is Zeus's domain and symbol?,Zeus rules the sky and thunder. His symbols ar...,"Zeus's domain is the sky, thunder, lightning, ...","Greek Mythology: Gods, Titans & Major Events\n..."
2,Who are the children of Kronos and Rhea?,"Hestia, Demeter, Hera, Hades, Poseidon, and Zeus.","The children of Kronos and Rhea are Zeus, Hest...","Greek Mythology: Gods, Titans & Major Events\n..."
3,Why did Kronos swallow his children?,He feared a prophecy that one of his children ...,Kronos swallowed his children because he was h...,(ash nymphs). From the sea-foam that gathered ...
4,What is the Titanomachy?,The Titanomachy was a ten-year war between the...,The Titanomachy is the ten-year war between th...,"Greek Mythology: Gods, Titans & Major Events\n..."


In [3]:
df = pd.read_csv("Complete_Eval_dataset.csv")
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head()

,query,ground_truth,response,reference
0,Who is the king of the Olympian gods?,Zeus is the king of the Olympian gods.,The king of the Olympian gods is Zeus.,denying his divinity — King Pentheus of Thebes...
1,What is Zeus's domain and symbol?,Zeus rules the sky and thunder. His symbols ar...,"Zeus's domain is the sky, thunder, lightning, ...","Greek Mythology: Gods, Titans & Major Events\n..."
2,Who are the children of Kronos and Rhea?,"Hestia, Demeter, Hera, Hades, Poseidon, and Zeus.","The children of Kronos and Rhea are Zeus, Hest...","Greek Mythology: Gods, Titans & Major Events\n..."
3,Why did Kronos swallow his children?,He feared a prophecy that one of his children ...,Kronos swallowed his children because he was h...,(ash nymphs). From the sea-foam that gathered ...
4,What is the Titanomachy?,The Titanomachy was a ten-year war between the...,The Titanomachy is the ten-year war between th...,"Greek Mythology: Gods, Titans & Major Events\n..."


In [4]:
df.rename(columns={"query": "user_input","reference":"retrieved_contexts", "ground_truth": 'reference'}, inplace=True)
df.head()

,user_input,reference,response,retrieved_contexts
0,Who is the king of the Olympian gods?,Zeus is the king of the Olympian gods.,The king of the Olympian gods is Zeus.,denying his divinity — King Pentheus of Thebes...
1,What is Zeus's domain and symbol?,Zeus rules the sky and thunder. His symbols ar...,"Zeus's domain is the sky, thunder, lightning, ...","Greek Mythology: Gods, Titans & Major Events\n..."
2,Who are the children of Kronos and Rhea?,"Hestia, Demeter, Hera, Hades, Poseidon, and Zeus.","The children of Kronos and Rhea are Zeus, Hest...","Greek Mythology: Gods, Titans & Major Events\n..."
3,Why did Kronos swallow his children?,He feared a prophecy that one of his children ...,Kronos swallowed his children because he was h...,(ash nymphs). From the sea-foam that gathered ...
4,What is the Titanomachy?,The Titanomachy was a ten-year war between the...,The Titanomachy is the ten-year war between th...,"Greek Mythology: Gods, Titans & Major Events\n..."


In [5]:
df["retrieved_contexts"] = df["retrieved_contexts"].apply(lambda x: [x] if isinstance(x, str) else x)


# RAG Evaluation

In [6]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import GPTModel
from deepeval.evaluate import AsyncConfig, DisplayConfig
import time

In [ ]:
os.environ["DEEPEVAL_ASYNC_TIMEOUT"] = "120"


In [7]:
test_cases = []

In [8]:
for i in range(len(df)):
    test_cases.append(
        LLMTestCase(
            input=df["user_input"].iloc[i],
            actual_output=df["response"].iloc[i],
            retrieval_context=df["retrieved_contexts"].iloc[i],  
            expected_output=df["reference"].iloc[i],
        )
    )


In [9]:
metrics = [
    FaithfulnessMetric(),
    AnswerRelevancyMetric()
]

In [10]:
len(test_cases)

49

In [11]:
all_results = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/50...", end=" ")
    try:
        res = evaluate(
            test_cases=[tc],
            metrics=metrics,
            async_config=AsyncConfig(run_async=False),
            display_config=DisplayConfig(show_indicator=False)  
        )
        all_results.append(res)
        print("✓")
    except Exception as e:
        print(f"✗ Failed: {e}")
        all_results.append(None)
    
    time.sleep(1) 

print("Done!")


Evaluating 1/50... 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Faithfulness                          │ 1.00                           │ 100.00%               │ 1             │
│  Answer Relevancy                      │ 1.00                           │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=2772474;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 26.26s | token cost: 0.02987 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✓
Evaluating 2/50... 

...

✓
Evaluating 49/50... 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Faithfulness                          │ 1.00                           │ 100.00%               │ 1             │
│  Answer Relevancy                      │ 1.00                           │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=2772570;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 25.51s | token cost: 0.0319725 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✓
Done!


In [13]:
all_results[1]

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions, so the actual output appears fully consistent with the retrieval context—nice work.', strict_mode=False, evaluation_model='gpt-5.4', error=None, evaluation_cost=0.027169999999999996, verbose_logs='Truths (limit=None):\n[\n    "After the victory in the Titanomachy, Zeus and his siblings established a new divine order centred on Mount Olympus.",\n    "Mount Olympus is described as the highest mountain in Greece.",\n    "The roster of the twelve Olympians varied slightly across city-states and time periods.",\n    "The twelve Olympians became recipients of formal public worship.",\n    "Each Olympian god maintained sacred temples, annual festivals, sacrificial rites, and mystery cults.",\n    "The myths of the Olympian gods form the heart of Greek religious and cultur

In [25]:
rows = []

for eval_result in all_results:
    for test_result in eval_result.test_results:
        row = {
            "name": test_result.name,
            "input": test_result.input,
            "actual_output": test_result.actual_output,
            "expected_output": test_result.expected_output,
            "success": test_result.success,
        }
        for metric in test_result.metrics_data:
            metric_name = metric.name.lower().replace(" ", "_")
            row[f"{metric_name}_score"] = metric.score
            row[f"{metric_name}_success"] = metric.success
            row[f"{metric_name}_reason"] = metric.reason
            row[f"{metric_name}_cost"] = metric.evaluation_cost
        
        rows.append(row)

results_df = pd.DataFrame(rows)
print(results_df.shape)
results_df.head()

(49, 13)


,name,input,actual_output,expected_output,success,faithfulness_score,faithfulness_success,faithfulness_reason,faithfulness_cost,answer_relevancy_score,answer_relevancy_success,answer_relevancy_reason,answer_relevancy_cost
0,test_case_0,Who is the king of the Olympian gods?,The king of the Olympian gods is Zeus.,Zeus is the king of the Olympian gods.,True,1.0,True,The score is 1.00 because there are no contrad...,0.025893,1.0,True,The score is 1.00 because the response directl...,0.003978
1,test_case_0,What is Zeus's domain and symbol?,"Zeus's domain is the sky, thunder, lightning, ...",Zeus rules the sky and thunder. His symbols ar...,True,1.0,True,The score is 1.00 because there are no contrad...,0.027170,1.0,True,The score is 1.00 because the answer is fully ...,0.004595
2,test_case_0,Who are the children of Kronos and Rhea?,"The children of Kronos and Rhea are Zeus, Hest...","Hestia, Demeter, Hera, Hades, Poseidon, and Zeus.",True,1.0,True,The score is 1.00 because there are no contrad...,0.029067,1.0,True,The score is 1.00 because the response fully a...,0.004428
3,test_case_0,Why did Kronos swallow his children?,Kronos swallowed his children because he was h...,He feared a prophecy that one of his children ...,True,1.0,True,The score is 1.00 because there are no contrad...,0.024977,1.0,True,The score is 1.00 because the response directl...,0.005380
4,test_case_0,What is the Titanomachy?,The Titanomachy is the ten-year war between th...,The Titanomachy was a ten-year war between the...,True,1.0,True,The score is 1.00 because there are no contrad...,0.030103,1.0,True,The score is 1.00 because the response directl...,0.005647


In [26]:
results_df['success'].value_counts()

success
True     48
False     1
Name: count, dtype: int64

In [23]:
df.to_csv("Evalution Scores", index=False)

In [27]:
results_df.drop(columns=['name', 'faithfulness_cost', "answer_relevancy_cost"], inplace=True)

In [28]:
results_df.to_csv("Cleaned_Eval_data.csv", index=False)